# 02 — Model Training

Reads: `data/train/1900_2021_DISASTERS.xlsx - train data.csv` (train split **only**)  
Evaluates on: `data/test/1970-2021_DISASTERS.xlsx - test data.csv` (holdout — never trained on)

Outputs (`backend/saved_models/`):
- `disaster_predictor.pkl` — dict bundle: XGBClassifier + LabelEncoders + region_freq_map
- `impact_regressor.pkl`  — dict of 4 regressors (XGB: deaths/damage, RF: injuries/affected)
- `shap_explainer.pkl`    — cached `shap.TreeExplainer` for the XGBClassifier

**NEVER use mean anywhere. Regressors trained on `log1p(target)` — `predictor.py` must apply `np.expm1()` at inference.**

In [ ]:
# ── Cell 1: Verify dependencies ──────────────────────────────────────────────
import subprocess, sys

pkgs = ["xgboost", "scikit-learn", "shap", "joblib", "pandas", "numpy"]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q"] + pkgs,
    stdout=subprocess.DEVNULL,
)
print("All dependencies ready.")

In [ ]:
# ── Cell 2: Imports & constants ───────────────────────────────────────────────
import re
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import joblib
import shap

from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import classification_report, f1_score
from xgboost import XGBClassifier, XGBRegressor

import xgboost, sklearn
print(f"XGBoost {xgboost.__version__} | sklearn {sklearn.__version__} | shap {shap.__version__}")


def parse_coord(val) -> float:
    """Parse lat/lon that may be numeric OR directional notation ('34.01 N', '78.46 W').
    Handles edge cases from the EM-DAT CSV: trailing periods ('78.9.'), mixed formats.
    Returns float, or np.nan if unparseable.
    """
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return np.nan
    s = str(val).strip().rstrip(".")   # strip trailing periods like '78.9.'
    # Try plain float first
    try:
        return float(s)
    except ValueError:
        pass
    # Try directional notation: '34.01 N', '78.46 W', '35.28 S', '31.15 E'
    m = re.match(r"^([0-9.]+)\s*([NSEWnsew])\s*$", s)
    if m:
        num_str, direction = m.group(1).rstrip("."), m.group(2).upper()
        try:
            num = float(num_str)
            return -num if direction in ("S", "W") else num
        except ValueError:
            pass
    return np.nan


# Only these 8 types are modelled — must match emdat_lookup.py VALID_DISASTER_TYPES exactly
VALID_DISASTER_TYPES = [
    "Flood",
    "Storm",
    "Earthquake",
    "Wildfire",
    "Volcanic activity",
    "Landslide",
    "Drought",
    "Extreme temperature",
]

TRAIN_CSV  = Path("../data/train/1900_2021_DISASTERS.xlsx - train data.csv")
TEST_CSV   = Path("../data/test/1970-2021_DISASTERS.xlsx - test data.csv")
MODELS_DIR = Path("../backend/saved_models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Feature vector order — MUST match predictor.py run_prediction() exactly.
# Any change here must be mirrored in backend/ml/predictor.py FEATURE_NAMES.
FEATURE_NAMES = [
    "latitude",
    "longitude",
    "continent_enc",
    "region_enc",
    "month",
    "dis_mag_value",
    "has_magnitude",
    "historical_freq",
    "decade",
    "day_offset",      # 0 for single predictions; 0–29 for 30-day forecast
]

print(f"Model output dir: {MODELS_DIR.resolve()}")
print(f"Feature count: {len(FEATURE_NAMES)}")


In [ ]:
# ── Cell 3: Load & filter train CSV ──────────────────────────────────────────
df = pd.read_csv(TRAIN_CSV, encoding="latin-1")
df["Disaster Type"] = df["Disaster Type"].str.strip()  # strip trailing spaces (CSV quirk)

df = df[df["Disaster Type"].isin(VALID_DISASTER_TYPES)].copy()
df = df.reset_index(drop=True)

print(f"Train rows after filtering to 8 valid types: {len(df):,}")
print()
print(df["Disaster Type"].value_counts().to_string())

In [ ]:
# ── Cell 4: Parse & impute Latitude / Longitude ───────────────────────────────
# 274 rows store coordinates as directional strings ('34.01 N', '78.46 W') — parse first.
# Some rows have clearly invalid floats (e.g. 36100.0 — DMS artefacts) — mask before imputation.
# 83 % of rows have null lat/lon — impute: country median → continent median → global.

VALID_LAT = (-90.0,  90.0)
VALID_LON = (-180.0, 180.0)

for col, (lo, hi) in [("Latitude", VALID_LAT), ("Longitude", VALID_LON)]:
    df[col] = df[col].apply(parse_coord)
    out_of_range = (df[col] < lo) | (df[col] > hi)
    df.loc[out_of_range, col] = np.nan          # treat DMS artefacts as missing
    df[col] = df[col].fillna(df.groupby("Country")[col].transform("median"))
    df[col] = df[col].fillna(df.groupby("Continent")[col].transform("median"))
    df[col] = df[col].fillna(df[col].median())

assert df["Latitude"].isna().sum()  == 0, "Latitude still has nulls after imputation"
assert df["Longitude"].isna().sum() == 0, "Longitude still has nulls after imputation"
print("Lat/Lon parsed & imputed — 0 nulls remaining.")
print(f"  Lat range: [{df['Latitude'].min():.1f}, {df['Latitude'].max():.1f}]")
print(f"  Lon range: [{df['Longitude'].min():.1f}, {df['Longitude'].max():.1f}]")


In [ ]:
# ── Cell 5: Feature engineering (train) ──────────────────────────────────────

# --- Latitude / Longitude (lowercase for feature matrix) ---
df["latitude"]  = df["Latitude"]
df["longitude"] = df["Longitude"]

# --- Month (impute missing with per-type mode, fallback 6 = June) ---
month_mode_map = (
    df.dropna(subset=["Start Month"])
    .groupby("Disaster Type")["Start Month"]
    .agg(lambda x: int(x.mode().iloc[0]))
    .to_dict()
)
df["month"] = df["Start Month"].copy()
for dtype, mode_val in month_mode_map.items():
    mask = df["month"].isna() & (df["Disaster Type"] == dtype)
    df.loc[mask, "month"] = mode_val
df["month"] = df["month"].fillna(6).astype(int)

# --- Decade ---
df["decade"] = (df["Year"] // 10) * 10

# --- Historical frequency: total recorded events per EM-DAT Region in TRAIN data ---
# Does not require knowing disaster type at inference — region-level aggregation.
region_freq_map = df.groupby("Region").size().to_dict()
df["historical_freq"] = df["Region"].map(region_freq_map).fillna(1).astype(int)

# --- Disaster magnitude ---
df["has_magnitude"] = df["Dis Mag Value"].notna().astype(int)
df["dis_mag_value"] = df["Dis Mag Value"].fillna(0.0)

# --- Categorical encoders (fit on TRAIN only — never on test) ---
le_continent = LabelEncoder()
le_region    = LabelEncoder()
le_target    = LabelEncoder()

df["continent_enc"] = le_continent.fit_transform(df["Continent"])
df["region_enc"]    = le_region.fit_transform(df["Region"])

# day_offset is 0 for all training rows (30-day forecast uses 0–29 at inference)
df["day_offset"] = 0

print(f"Continent classes: {len(le_continent.classes_)} → {list(le_continent.classes_)}")
print(f"Region classes: {len(le_region.classes_)}")
print(f"Month mode map: {month_mode_map}")

In [ ]:
# ── Cell 6: Build feature matrix + classification target ──────────────────────
X_train     = df[FEATURE_NAMES].values.astype(np.float32)
y_cls_train = le_target.fit_transform(df["Disaster Type"])

print(f"X_train shape: {X_train.shape}")
print(f"Classes ({len(le_target.classes_)}): {list(le_target.classes_)}")
print()
print("Class distribution (train):")
for i, cls in enumerate(le_target.classes_):
    count = int((y_cls_train == i).sum())
    print(f"  {cls:22s}: {count:,}")

print()
# Sanity: no NaN in feature matrix
nan_count = np.isnan(X_train).sum()
assert nan_count == 0, f"Feature matrix has {nan_count} NaN values!"
print("Feature matrix: 0 NaN values — OK")

In [ ]:
# ── Cell 7: Regression targets (log1p transform) ─────────────────────────────
# Disaster impact data is heavily right-skewed (e.g., Flood mean deaths = 1,735 vs median = 16).
# log1p transform compresses extreme outliers so tree regressors generalise better.
# IMPORTANT: predictor.py must apply np.expm1() to all regressor outputs before returning.

for col in [
    "Total Deaths",
    "No Injured",
    "No Affected",
    "Total Damages ('000 US$)",
]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).clip(lower=0)

y_deaths_log   = np.log1p(df["Total Deaths"].values)
y_injuries_log = np.log1p(df["No Injured"].values)
y_affected_log = np.log1p(df["No Affected"].values)
y_damage_log   = np.log1p(df["Total Damages ('000 US$)"].values)

print("Regression targets (raw values, median / max):")
print(f"  Deaths   — median: {int(np.median(df['Total Deaths'])):>8,}  max: {int(df['Total Deaths'].max()):>10,}")
print(f"  Injuries — median: {int(np.median(df['No Injured'])):>8,}  max: {int(df['No Injured'].max()):>10,}")
print(f"  Affected — median: {int(np.median(df['No Affected'])):>8,}  max: {int(df['No Affected'].max()):>10,}")
print(f"  Damage   — median: {int(np.median(df["Total Damages ('000 US$)"])):>8,}  max: {int(df["Total Damages ('000 US$)"].max()):>10,}  ('000 USD)")
print()
print("log1p-transformed (what regressors are trained on):")
print(f"  Deaths   — median log: {np.median(y_deaths_log):.3f}")
print(f"  Injuries — median log: {np.median(y_injuries_log):.3f}")
print(f"  Affected — median log: {np.median(y_affected_log):.3f}")
print(f"  Damage   — median log: {np.median(y_damage_log):.3f}")

In [ ]:
# ── Cell 8: Load & preprocess test CSV (holdout evaluation) ──────────────────
# Uses FITTED encoders from train — never re-fits on test data.
# Unknown continent/region values are mapped to 0 (most-common class).

def safe_encode(le: LabelEncoder, values) -> np.ndarray:
    """Encode with a fitted LabelEncoder; map unseen labels to 0."""
    known = set(le.classes_)
    return np.array(
        [le.transform([v])[0] if v in known else 0 for v in values],
        dtype=np.int32,
    )


def preprocess_test(df_raw: pd.DataFrame) -> pd.DataFrame:
    df_t = df_raw.copy()
    df_t["Disaster Type"] = df_t["Disaster Type"].str.strip()
    df_t = df_t[df_t["Disaster Type"].isin(VALID_DISASTER_TYPES)].reset_index(drop=True)

    # Parse directional coords, mask out-of-range, then impute (same strategy as train)
    for col, (lo, hi) in [("Latitude", (-90.0, 90.0)), ("Longitude", (-180.0, 180.0))]:
        df_t[col] = df_t[col].apply(parse_coord)
        out_of_range = (df_t[col] < lo) | (df_t[col] > hi)
        df_t.loc[out_of_range, col] = np.nan
        df_t[col] = df_t[col].fillna(df_t.groupby("Country")[col].transform("median"))
        df_t[col] = df_t[col].fillna(df_t.groupby("Continent")[col].transform("median"))
        df_t[col] = df_t[col].fillna(df_t[col].median())

    df_t["latitude"]  = df_t["Latitude"]
    df_t["longitude"] = df_t["Longitude"]
    df_t["month"]     = df_t["Start Month"].fillna(6).astype(int)
    df_t["decade"]    = (df_t["Year"] // 10) * 10

    # Use TRAIN's region_freq_map — unknown regions get freq = 1
    df_t["historical_freq"] = df_t["Region"].map(region_freq_map).fillna(1).astype(int)

    df_t["has_magnitude"] = df_t["Dis Mag Value"].notna().astype(int)
    df_t["dis_mag_value"] = pd.to_numeric(df_t["Dis Mag Value"], errors="coerce").fillna(0.0)

    # Encode with TRAIN encoders only
    df_t["continent_enc"] = safe_encode(le_continent, df_t["Continent"])
    df_t["region_enc"]    = safe_encode(le_region,    df_t["Region"])
    df_t["day_offset"]    = 0

    return df_t


df_test_raw = pd.read_csv(TEST_CSV, encoding="latin-1")
df_test     = preprocess_test(df_test_raw)

X_test = df_test[FEATURE_NAMES].values.astype(np.float32)

# Encode test labels — filter any type not in training classes (shouldn't happen, just safety)
known_types = set(le_target.classes_)
mask_known  = df_test["Disaster Type"].isin(known_types)
X_test_eval     = X_test[mask_known]
y_cls_test_eval = le_target.transform(df_test.loc[mask_known, "Disaster Type"])

print(f"Test rows (filtered to 8 types): {len(df_test):,}")
print(f"Test rows usable for eval (known labels): {X_test_eval.shape[0]:,}")
print(f"X_test shape: {X_test_eval.shape}")
nan_test = np.isnan(X_test_eval).sum()
assert nan_test == 0, f"Test feature matrix has {nan_test} NaN values!"
print("Test feature matrix: 0 NaN values — OK")


In [ ]:
# ── Cell 9: Train XGBoostClassifier ──────────────────────────────────────────
%%time
clf = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
clf.fit(X_train, y_cls_train)

print("Classifier trained.")
print()
print("Feature importances:")
for name, imp in sorted(
    zip(FEATURE_NAMES, clf.feature_importances_),
    key=lambda x: x[1], reverse=True,
):
    bar = "#" * int(imp * 40)
    print(f"  {name:20s}  {imp:.4f}  {bar}")


In [ ]:
# ── Cell 10: Train 4 impact regressors ───────────────────────────────────────
# XGBoost → estimated_deaths, estimated_damage_usd  (spec: CLAUDE.md ML section)
# Random Forest → estimated_injuries, estimated_affected
# All targets are log1p-transformed — predictor.py applies np.expm1() at inference.
%%time

xgb_deaths = XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

xgb_damage = XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

rf_injuries = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)

rf_affected = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)

xgb_deaths.fit(X_train,  y_deaths_log)
xgb_damage.fit(X_train,  y_damage_log)
rf_injuries.fit(X_train, y_injuries_log)
rf_affected.fit(X_train, y_affected_log)

print("All 4 regressors trained.")

In [ ]:
# ── Cell 11: Evaluate on holdout test set ────────────────────────────────────
y_pred_cls = clf.predict(X_test_eval)

print("=" * 65)
print("CLASSIFICATION REPORT — holdout test set (1970–2021)")
print("=" * 65)
print(classification_report(
    y_cls_test_eval,
    y_pred_cls,
    target_names=le_target.classes_,
    zero_division=0,
))

macro_f1   = f1_score(y_cls_test_eval, y_pred_cls, average="macro",   zero_division=0)
weighted_f1 = f1_score(y_cls_test_eval, y_pred_cls, average="weighted", zero_division=0)
print(f"Macro F1   : {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

print()
print("=" * 65)
print("REGRESSION SANITY CHECK — median predicted vs actual (train sample)")
print("=" * 65)
rng = np.random.default_rng(42)
idx = rng.choice(len(X_train), min(2000, len(X_train)), replace=False)
X_s = X_train[idx]

# Apply expm1 to reverse the log1p transform used during training
pred_deaths   = np.expm1(xgb_deaths.predict(X_s)).clip(min=0)
pred_injuries = np.expm1(rf_injuries.predict(X_s)).clip(min=0)
pred_affected = np.expm1(rf_affected.predict(X_s)).clip(min=0)
pred_damage   = np.expm1(xgb_damage.predict(X_s)).clip(min=0)

actual_deaths   = df["Total Deaths"].values[idx]
actual_injuries = df["No Injured"].values[idx]
actual_affected = df["No Affected"].values[idx]
actual_damage   = df["Total Damages ('000 US$)"].values[idx]

def fmt(n): return f"{int(n):>12,}"

print(f"{'Metric':<12}  {'Actual median':>14}  {'Predicted median':>16}")
print("-" * 46)
print(f"{'Deaths':<12}  {fmt(np.median(actual_deaths))}  {fmt(np.median(pred_deaths))}")
print(f"{'Injuries':<12}  {fmt(np.median(actual_injuries))}  {fmt(np.median(pred_injuries))}")
print(f"{'Affected':<12}  {fmt(np.median(actual_affected))}  {fmt(np.median(pred_affected))}")
print(f"{'Damage(K$)':<12}  {fmt(np.median(actual_damage))}  {fmt(np.median(pred_damage))}")

In [ ]:
# ── Cell 12: Build & verify SHAP TreeExplainer ───────────────────────────────
# Instantiated ONCE here, saved to disk. predictor.py loads from pkl — never re-instantiates.
# Saving the explainer avoids ~200 ms startup cost per request if constructed per-call.
%%time

explainer = shap.TreeExplainer(clf)

# Smoke-test on a tiny batch
_shap_sample = explainer.shap_values(X_train[:20])
_shap_arr    = np.array(_shap_sample)
print(f"SHAP values shape: {_shap_arr.shape}")
# Expected: (20, n_features, n_classes)  — 3-D for multi-class XGBClassifier


def extract_top_shap(shap_values, class_idx: int) -> list:
    """Identical to predictor.py _extract_top_shap — tested here to catch shape issues."""
    arr  = np.array(shap_values)
    vals = np.abs(arr[0, :, class_idx]) if arr.ndim == 3 else np.abs(arr[0])
    total = vals.sum()
    if total == 0:
        return []
    ranked = sorted(zip(FEATURE_NAMES, vals), key=lambda x: x[1], reverse=True)[:3]
    return [{"feature": name, "contribution_pct": round(float(val / total * 100), 1)}
            for name, val in ranked]


sample_top3 = extract_top_shap(_shap_arr, class_idx=0)
print(f"Sample top-3 SHAP features: {sample_top3}")
print("SHAP explainer verified OK.")

In [ ]:
# ── Cell 13: Save all 3 model files ──────────────────────────────────────────

# --- disaster_predictor.pkl ---
# Bundled as a dict so predictor.py can access encoders + freq map from a single load.
classifier_bundle = {
    "model":              clf,
    "le_continent":       le_continent,     # LabelEncoder fitted on train Continent
    "le_region":          le_region,         # LabelEncoder fitted on train Region
    "le_target":          le_target,         # LabelEncoder for disaster type classes
    "region_freq_map":    region_freq_map,   # {region_name: int} total events in train
    "feature_names":      FEATURE_NAMES,     # exact feature order used at training
    "targets_are_log1p":  True,              # regressors output log1p — apply expm1 at inference
}
joblib.dump(classifier_bundle, MODELS_DIR / "disaster_predictor.pkl")
size_clf = (MODELS_DIR / "disaster_predictor.pkl").stat().st_size
print(f"  disaster_predictor.pkl  {size_clf / 1024:>8.0f} KB")

# --- impact_regressor.pkl ---
impact_regressors = {
    "deaths":   xgb_deaths,
    "injuries": rf_injuries,
    "affected": rf_affected,
    "damage":   xgb_damage,
}
joblib.dump(impact_regressors, MODELS_DIR / "impact_regressor.pkl")
size_reg = (MODELS_DIR / "impact_regressor.pkl").stat().st_size
print(f"  impact_regressor.pkl    {size_reg / 1024:>8.0f} KB")

# --- shap_explainer.pkl ---
joblib.dump(explainer, MODELS_DIR / "shap_explainer.pkl")
size_shap = (MODELS_DIR / "shap_explainer.pkl").stat().st_size
print(f"  shap_explainer.pkl      {size_shap / 1024:>8.0f} KB")

total_mb = (size_clf + size_reg + size_shap) / 1024 / 1024
print()
print(f"Total size: {total_mb:.1f} MB")
print(f"Saved to: {MODELS_DIR.resolve()}")

In [ ]:
# ── Cell 14: End-to-end smoke test (simulates predictor.py inference) ─────────
# Reload from disk to verify pkl round-trip is correct.

_bundle    = joblib.load(MODELS_DIR / "disaster_predictor.pkl")
_regs      = joblib.load(MODELS_DIR / "impact_regressor.pkl")
_explainer = joblib.load(MODELS_DIR / "shap_explainer.pkl")

# Simulate a request: Egypt, Flood region, month=7 (July)
egypt_continent = "Africa"
egypt_region    = "Northern Africa"

_le_c    = _bundle["le_continent"]
_le_r    = _bundle["le_region"]
_le_t    = _bundle["le_target"]
_freq_m  = _bundle["region_freq_map"]

cont_enc  = _le_c.transform([egypt_continent])[0]
reg_enc   = _le_r.transform([egypt_region])[0]
hist_freq = _freq_m.get(egypt_region, 1)

X_demo = np.array([[30.0, 31.0, cont_enc, reg_enc, 7, 0.0, 0, hist_freq, 2020, 0]], dtype=np.float32)
# order: latitude, longitude, continent_enc, region_enc, month, dis_mag_value, has_magnitude, historical_freq, decade, day_offset

proba      = _bundle["model"].predict_proba(X_demo)[0]
class_idx  = int(np.argmax(proba))
dtype_pred = _le_t.classes_[class_idx]
prob_pred  = float(proba[class_idx])

deaths_pred   = int(np.expm1(_regs["deaths"].predict(X_demo)[0]).clip(min=0))
injuries_pred = int(np.expm1(_regs["injuries"].predict(X_demo)[0]).clip(min=0))
affected_pred = int(np.expm1(_regs["affected"].predict(X_demo)[0]).clip(min=0))
damage_pred   = int(np.expm1(_regs["damage"].predict(X_demo)[0]).clip(min=0))

shap_vals   = _explainer.shap_values(X_demo)
shap_arr    = np.array(shap_vals)
top3        = extract_top_shap(shap_arr, class_idx)

print("── Smoke-test inference (Egypt, July 2020) ──")
print(f"  Predicted disaster type : {dtype_pred}")
print(f"  Probability             : {prob_pred:.4f}")
print(f"  Estimated deaths        : {deaths_pred:,}")
print(f"  Estimated injuries      : {injuries_pred:,}")
print(f"  Estimated affected      : {affected_pred:,}")
print(f"  Estimated damage ('000$): {damage_pred:,}")
print(f"  Top-3 SHAP features     : {top3}")
print()
print("All pkl files load and run correctly. Phase 3 backend is ready to use these models.")

## Next steps

1. Upload pkl files to HuggingFace Hub so Render can download at startup:
   ```bash
   pip install huggingface_hub
   huggingface-cli login
   huggingface-cli upload <HUGGINGFACE_REPO_ID> backend/saved_models/ .
   ```

2. Implement `backend/ml/predictor.py` using the bundle dict loaded from `disaster_predictor.pkl`.
   - `_bundle["model"]` → XGBClassifier  
   - `_bundle["le_continent"]` / `le_region` / `le_target` → encoders  
   - `_bundle["region_freq_map"]` → historical_freq lookup  
   - `_bundle["targets_are_log1p"]` → must apply `np.expm1()` to all regressor outputs  

3. Implement `backend/services/predictor_service.py` — calls `predictor.run_prediction()`,
   then `emdat_lookup.resolve_impact_stats()` for median-based impact stats,
   then saves to `predictions` table.